# TP2 (STARTER): Grounded Agentic RAG, Embeddings & The Gemini Interactions API

**Course:** `4S4TWS_21` — Agentic AI (École Centrale de Lyon & emlyon business school)  
**Instructor:** Chouaieb Nemri  
**Session:** Day 1 Afternoon — Wednesday, October 14, 2026 (`14:15 – 18:15`)  
**SDK & Models:** `google-genai` (`gemma-4-26b-a4b-it` / `gemma-4-31b-it` + `gemini-embedding-2`)

---

## Learning Objectives
1. Understand why **Naive Single-Shot RAG** fails on complex corporate sustainability claims and how **Agentic RAG** turns retrieval into a multi-step reasoning tool.
2. Build a semantic search index over `green_claims_corpus.json` using **`gemini-embedding-2`** (with a deterministic local TF-IDF/keyword cosine fallback).
3. Use the **Gemini Interactions API (`client.interactions.create`)** with `previous_interaction_id` for stateful multi-turn sessions (`interaction.output_text`).
4. Produce a citation-grounded **Anti-Greenwashing Audit Verdict** that flags unverified offset claims under the **EU Green Claims Directive (`REG-EU-GCD-01`)**.

In [ ]:
import os
import json
import math
import re
import time
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field
try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None

CORPUS_PATHS = [
    Path("../data/green_claims_corpus.json"),
    Path("03_Labs_TP/data/green_claims_corpus.json"),
    Path("/Users/nemri/Downloads/ecl-agentic-ai-course/03_Labs_TP/data/green_claims_corpus.json"),
]
for p in CORPUS_PATHS:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            CORPUS_DATA = json.load(f)
        break

DOCUMENTS = CORPUS_DATA["documents"]
MODEL_ID = "gemma-4-26b-a4b-it"  # Or "gemma-4-31b-it"
print(f"Loaded {len(DOCUMENTS)} regulatory and corporate disclosure documents. Model: {MODEL_ID}")

---
## Exercise 1: Semantic Retrieval Tool with `gemini-embedding-2`

In Agentic RAG, the retriever is exposed as a **tool** (`search_sustainability_corpus(query: str, category_filter: str = "ALL")`). The agent can call it multiple times—first to look up the corporate claim, and second to look up the governing EU regulation!


In [ ]:
# TODO: Implement search_sustainability_corpus(query: str, category_filter: str = "ALL") -> dict
# and evaluate_eu_green_claim_rule(...) -> dict


---
## Exercise 2: Stateful Verification with the Interactions API & Grounded ReAct Loop

The `google-genai` SDK provides the **Interactions API** (`client.interactions.create`), which:
- Stores conversation state server-side (`previous_interaction_id=interaction1.id`)
- Returns the model's response directly via `interaction.output_text`

For multi-step tool verification + structured Pydantic extraction (`GreenwashingAuditVerdict`), use the 2-stage ReAct pattern and `strip_thought_parts(...)` below so your Free-Tier API key stays well inside the 16,000 TPM budget even during batch audits!

In [ ]:
def strip_thought_parts(content):
    """Strip internal `thought=True` blocks before appending to `contents` (~65% Free-Tier input token savings)."""
    if content is None or not getattr(content, "parts", None):
        return content
    clean_parts = [p for p in content.parts if not getattr(p, "thought", False)]
    return types.Content(role=content.role, parts=clean_parts)

def generate_with_retry(client, *, model: str, contents, config=None, max_retries: int = 4):
    """Automatically pause and retry if a batch loop reaches the Free-Tier 16k TPM window."""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except Exception as exc:
            msg = str(exc)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) and attempt < max_retries - 1:
                m = re.search(r"retry in (\d+)", msg, flags=re.IGNORECASE)
                wait_s = int(m.group(1)) + 3 if m else 30
                print(f"  [Free-Tier Cooldown] Pausing {wait_s}s for TPM reset...")
                time.sleep(wait_s)
                continue
            raise

# TODO: Define GreenwashingAuditVerdict and implement audit_corporate_claim(company_query: str)

---
## Stretch Exercise (Bonus for Fast Finishers): Batch Audit Table Across Corporate Claims

Run `audit_corporate_claim` across multiple corporate claims in `green_claims_corpus.json` (`AeroTransit`, `NordicVolt`, `VelvetLoop`, `RhôneAlpin`, `HyperCompute`) and print a clean summary table showing which claims are **COMPLIANT** vs. **NON_COMPLIANT_GREENWASHING** along with their verified citation IDs.


In [ ]:
# TODO (Stretch Task): Loop over the corporate claims and print a clean compliance summary table
